In [ ]:
import os

import pandas as pd
import dask.dataframe as dd
from dask.distributed import LocalCluster, Client

from utilities import walk_through_files

cluster = LocalCluster(n_workers=4)
client = Client(cluster)

In [ ]:
HOMECOUNT_PATH = os.path.join("intermediate-output", "device-home-summary.csv.gz")
ALTERNATE_DUMP_PATH = os.path.join("raw-data", "datadump")
SAVEPATH = os.path.join("intermediate-output", "daily-movecount-strict.csv.gz")

MIN_MODAL_COUNT = 30
MIN_MODAL_SHARE = 0.50

In [ ]:
df_home = (
    dd.read_csv(HOMECOUNT_PATH, compression="gzip")
    .query(
        f"modal_count > {MIN_MODAL_COUNT} "
        f"and modal_share > {MIN_MODAL_SHARE}"
    )
    .rename(columns={"modal_area_id": "home", "modal_count": "count"})
    [["device_xid", "home", "count"]]
)

In [ ]:
file_lists = tuple(walk_through_files(ALTERNATE_DUMP_PATH, "csv.gz", fullpath=True))

df_daily = dd.read_csv(
    file_lists, compression="gzip", usecols=["device_xid", "date", "area_id"]
)

df_gravity = (
    df_daily
    .merge(df_home, how="left", on="device_xid")
    .groupby(["date", "home", "area_id"])
    .size()
    .reset_index()
    .rename(columns={0: "inflow_count"})
    .astype({"home": "int"})
    .compute()
)

df_gravity.to_csv(SAVEPATH, index=False, compression="gzip")